In [1]:
using Pkg
Pkg.status(; mode=Pkg.PKGMODE_MANIFEST)

Status `~/populi.Wk/DoubleNaught/Manifest.toml`
 ⌃ [992eb4ea] CondaPkg v0.2.36
   [9a962f9c] DataAPI v1.16.0
   [e2d170a0] DataValueInterfaces v1.0.0
   [82899510] IteratorInterfaceExtensions v1.0.0
   [692b3bcd] JLLWrappers v1.8.0
 ⌃ [682c06a0] JSON v1.7.0
   [1914dd2f] MacroTools v0.5.16
   [0b3b1443] MicroMamba v0.1.15
 ⌃ [bac558e1] OrderedCollections v2.0.1
 ⌅ [69de0a69] Parsers v2.8.7
   [fa939f87] Pidfile v1.3.0
 ⌃ [aea7be01] PrecompileTools v1.2.1
 ⌃ [21216c6a] Preferences v1.5.2
 ⌃ [6099a3de] PythonCall v0.9.35
   [6c6a2e73] Scratch v1.3.0
 ⌃ [ec057cc2] StructUtils v2.8.3
   [3783bdb8] TableTraits v1.0.1
 ⌃ [bd369af6] Tables v1.13.0
   [e17b2a0c] UnsafePointers v1.0.0
→⌃ [c8ffd9c3] MbedTLS_jll v2.28.6+0
   [f8abcde7] micromamba_jll v2.3.1+0
 ⌃ [4d7b5844] pixi_jll v0.63.2+0
   [0dad84c5] ArgTools v1.1.2
   [56f22d72] Artifacts v1.11.0
   [2a0f44e3] Base64 v1.11.0
   [ade2ca70] Dates v1.11.0
   [f43a241f] Downloads v1.6.0
   [7b1f6079] FileWatching v1.11.0
   [b77e0a4c] Interacti

In [2]:
println(pathof(D4M))
println(pathof(OntologyAnvil))
println(Base.active_project())

UndefVarError: UndefVarError: `D4M` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [3]:
Base.active_project()

"/Users/gcr/populi.Wk/DoubleNaught/Project.toml"

In [4]:
using D4M, DataFrames, DelimitedFiles, OntologyAnvil


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [5]:
using D4M
using OntologyAnvil

# Explicit (Row, Col, Val) triples -- one tuple per fact, unzipped below, so there
# is no way for row/col/val to drift out of alignment the way the earlier
# three-separate-arrays version did.
triples = [
    ("Prop",      "disjointWith|Character", "Character"),
    ("Prop",      "disjointWith|Creature",  "Creature"),
    ("Prop",      "requires|entity_name",   "entity_name"),
    ("Character", "disjointWith|Prop",      "Prop"),
    ("Character", "disjointWith|Creature",  "Creature"),
    ("Character", "requires|entity_name",   "entity_name"),
    ("Creature",  "disjointWith|Prop",      "Prop"),
    ("Creature",  "disjointWith|Character", "Character"),
    ("Creature",  "requires|entity_name",   "entity_name"),
]

row = [t[1] for t in triples]
col = [t[2] for t in triples]
val = [t[3] for t in triples]

@assert length(row) == length(col) == length(val)
println("triples: ", length(row))

ontology = Assoc(row, col, val)
engine = build_engine(ontology)

println("classes: ", engine.classes)
println("slots:   ", engine.slots)

# --- Case 1: a well-formed Prop (lamp), entity_name filled, no conflicting class ---
ok1, violations1 = validate_entity(engine, ["Prop"], ["entity_name"])
println("\nCase 1 (valid Prop): valid=$ok1  violations=$violations1")

# --- Case 2: a Prop missing its required entity_name slot ---
ok2, violations2 = validate_entity(engine, ["Prop"], String[])
println("Case 2 (Prop missing entity_name): valid=$ok2  violations=$violations2")

# --- Case 3: an entity mis-extracted as BOTH Prop and Character at once ---
ok3, violations3 = validate_entity(engine, ["Prop", "Character"], ["entity_name"])
println("Case 3 (Prop+Character, disjoint): valid=$ok3  violations=$violations3")

# --- Case 4: a standalone check, bypassing validate_entity's composition ---
ok4, violations4 = check_class_assignment(engine, ["Creature", "Character"])
println("Case 4 (Creature+Character, disjoint): valid=$ok4  violations=$violations4")

triples: 9
classes: ["Character", "Creature", "Prop"]
slots:   ["entity_name"]

Case 1 (valid Prop): valid=true  violations=String[]
Case 2 (Prop missing entity_name): valid=false  violations=["Missing required slot(s) for Prop: [\"entity_name\"]"]
Case 3 (Prop+Character, disjoint): valid=false  violations=["Disjointness violation: entity asserts mutually exclusive classes among [\"Character\", \"Prop\"]", "Cannot check slot requirements: entity must resolve to exactly one recognized class, got [\"Prop\", \"Character\"] from asserted classes [\"Prop\", \"Character\"]"]
Case 4 (Creature+Character, disjoint): valid=false  violations=["Disjointness violation: entity asserts mutually exclusive classes among [\"Character\", \"Creature\"]"]
